In [0]:
CREATE OR REPLACE TABLE ifood_case.silver.yellow_trips
USING DELTA
PARTITIONED BY (ano_mes)
AS
SELECT
    CAST(VendorID AS INT)                        AS vendor_id,
    tpep_pickup_datetime,
    tpep_dropoff_datetime,
    CAST(passenger_count AS INT)                 AS passenger_count,
    CAST(total_amount AS DECIMAL(10,2))          AS total_amount,
    DATE_FORMAT(tpep_pickup_datetime,'yyyy-MM')  AS ano_mes,
    HOUR(tpep_pickup_datetime)                   AS hora_pickup
FROM ifood_case.bronze.yellow_tripdata_raw
WHERE
    tpep_pickup_datetime >= '2023-01-01'
    AND tpep_pickup_datetime <  '2023-06-01'
    AND total_amount    IS NOT NULL
    AND total_amount    >= 0
    AND passenger_count IS NOT NULL
    AND passenger_count >  0
    AND tpep_dropoff_datetime > tpep_pickup_datetime;
    

CREATE TABLE IF NOT EXISTS ifood_case.bronze.quality_log (
    run_ts          TIMESTAMP,
    camada          STRING,
    total_linhas    LONG,
    removidos_valor LONG,
    removidos_pax   LONG,
    removidos_tempo LONG,
    removidos_data  LONG,
    pct_aproveitamento DOUBLE
);

INSERT INTO ifood_case.bronze.quality_log
WITH bronze AS (SELECT COUNT(*) AS n FROM ifood_case.bronze.yellow_tripdata_raw),
     silver AS (SELECT COUNT(*) AS n FROM ifood_case.silver.yellow_trips),
     violacoes AS (
         SELECT
             SUM(CASE WHEN total_amount    <  0    THEN 1 ELSE 0 END) AS v1,
             SUM(CASE WHEN passenger_count <= 0    THEN 1 ELSE 0 END) AS v2,
             SUM(CASE WHEN tpep_dropoff_datetime
                      <= tpep_pickup_datetime      THEN 1 ELSE 0 END) AS v3,
             SUM(CASE WHEN tpep_pickup_datetime < '2023-01-01'
                       OR tpep_pickup_datetime >= '2023-06-01'
                                                   THEN 1 ELSE 0 END) AS v4
         FROM ifood_case.bronze.yellow_tripdata_raw
     )
SELECT
    CURRENT_TIMESTAMP(),
    'silver',
    bronze.n,
    violacoes.v1,
    violacoes.v2,
    violacoes.v3,
    violacoes.v4,
    ROUND(silver.n * 100.0 / bronze.n, 2)
FROM bronze, silver, violacoes;

-- Consulta do historico
SELECT * FROM ifood_case.bronze.quality_log ORDER BY run_ts DESC;

run_ts,camada,total_linhas,removidos_valor,removidos_pax,removidos_tempo,removidos_data,pct_aproveitamento
2026-06-26T14:11:16.805Z,silver,16186386,141407,273481,6181,104,94.76


In [0]:
SELECT
    ano_mes,
    COUNT(*)                    AS corridas,
    ROUND(AVG(total_amount), 2) AS ticket_medio
FROM ifood_case.silver.yellow_trips
GROUP BY ano_mes
ORDER BY ano_mes;


ano_mes,corridas,ticket_medio
2023-01,2917665,27.46
2023-02,2764200,27.37
2023-03,3226999,28.28
2023-04,3109876,28.78
2023-05,3319397,29.45


In [0]:
SELECT
    SUM(CASE WHEN total_amount < 0 THEN 1 ELSE 0 END)
        AS viola_valor_negativo,
    SUM(CASE WHEN passenger_count <= 0 THEN 1 ELSE 0 END)
        AS viola_passageiros,
    SUM(CASE WHEN tpep_dropoff_datetime <= tpep_pickup_datetime
             THEN 1 ELSE 0 END)
        AS viola_tempo
FROM ifood_case.silver.yellow_trips;


viola_valor_negativo,viola_passageiros,viola_tempo
0,0,0
